In [ ]:
cell = "K562"
resolution = 10000    # Hi-C resolution (bp): 10000(10kb), 25000(25kb)
display_reso = int(resolution / 1000)
# Update this root_dir to your own project path before running
root_dir = "/mnt/d/TAD-MultiOmicsNet"

In [ ]:
#### Generate consensus TAD boundaries
# Callers: Arrowhead, DI, deDoc, SpectralTAD, TopDom

In [ ]:
"""
Generate consensus TAD boundaries.
Input: tab-separated file, first line is header "start end" (skipped).
2 columns: TAD start(bp), TAD end(bp).
Raw boundary coordinates are rounded to the nearest multiple of Hi-C bin resolution,
to handle positions that do not fall exactly on bin edges.
Output: space-separated boundary file, 2 columns: boundary start(bp), end(bp).
Consensus: boundaries detected by ≥2 of 5 TAD callers.
"""

import numpy as np
import pandas as pd
import os

def process_file(file_path):
    if not os.path.exists(file_path):
        return np.array([], dtype=np.int64)
    df = pd.read_csv(file_path, header=None, delimiter='\t', skiprows=1)
    df[0] = pd.to_numeric(df[0], errors='coerce')
    df[1] = pd.to_numeric(df[1], errors='coerce')
    df = df.dropna(subset=[0, 1])
    df[0] = df[0].astype(np.int64)
    df[1] = df[1].astype(np.int64)
    df_np = np.concatenate([df[0].values, df[1].values])
    df_np = np.unique(df_np)
    return np.sort(df_np)

if __name__ == "__main__":
    # List of TAD boundary callers
    callers = ['Arrowhead', 'DI', 'deDoc', 'SpectralTAD', 'TopDom']
    chrs = list(range(1, 23)) + ['X']
    for chr_id in chrs:
        result_dict = {}
        for caller in callers:
            file_path = f'{root_dir}/{cell}/{display_reso}kb/all_TADs/{caller}/{cell}_{display_reso}kb_{caller}.chr{chr_id}'
            result_dict[caller] = process_file(file_path)
        
        all_data = np.concatenate(list(result_dict.values()))
        unique_data, counts = np.unique(all_data, return_counts=True)
        shared_coords = unique_data[counts >= 2]
        print(f"chr{chr_id}, the number of boundary bins shared by at least two tools: {len(shared_coords)}")
        
        formatted_data = [[v, v + resolution] for v in shared_coords]
        new_lines = formatted_data

        output_dir = f'{root_dir}/{cell}/{display_reso}kb/boundary-generate/'
        os.makedirs(output_dir, exist_ok=True)
        out_text = '\n'.join([' '.join(map(str, line)) for line in new_lines])
        with open(f'{output_dir}chr{chr_id}_{cell}_{display_reso}kb_boundary.txt', 'w') as f:
            f.write(out_text)


In [ ]:
### chr1-22 Generation of K562 test samples for all positive and negative samples

In [ ]:
import numpy as np

def openreadtxt(file_name):
    with open(file_name, 'r') as file:
        file_data = file.readlines()
        data = [row.replace('\n', '').split() for row in file_data]
    return data

def read_hic_matrix_chunk(file_path, chunk_size=1000):
    # Read the Hi-C matrix text in blocks line by line
    with open(file_path, 'r') as f:
        while True:
            chunk_lines = f.readlines(chunk_size * 10240)
            if not chunk_lines:
                break
            for line in chunk_lines:
                line = line.strip()
                if not line:
                    continue
                row_arr = np.array(line.split(), dtype=np.float32)
                yield row_arr

if __name__ == '__main__':
    chrs = list(range(1, 23))
    pad_up, pad_down = 4, 5
    pad_left, pad_right = 4, 5
    win_half = 5

    for chr in chrs:
        all_data = []
        label_T = 1
        label_F = 0
        ww = 0

        boundary_rows = openreadtxt(f'{root_dir}/{cell}/{display_reso}kb/boundary-generate/chr{chr}_{cell}_{display_reso}kb_boundary.txt')
        n_bound = len(boundary_rows)

        matrix_rows = []
        print(f"Start chunked reading chr{chr} Hi-C matrix")
        for row in read_hic_matrix_chunk(f'{root_dir}/{cell}/{display_reso}kb/prepare_data/{cell}_{display_reso}kb_ICE.chr{chr}'):
            pad_row = np.pad(row, (pad_left, pad_right), mode='constant')
            matrix_rows.append(pad_row)
        big_len = len(matrix_rows)
        row_dim = matrix_rows[0].shape[0]

        # Add zero padding to top and bottom of matrix
        top_pad = np.zeros((pad_up, row_dim), dtype=np.float32)
        bottom_pad = np.zeros((pad_down, row_dim), dtype=np.float32)
        padded_matrix = np.vstack([top_pad, np.vstack(matrix_rows), bottom_pad])
        print(f"Number of rows in the original matrix: {big_len}, matrix shape after padding: {padded_matrix.shape}")

        # m starts from 5
        for m in range(5, 5 + big_len):
            if ww < n_bound:
                bp_pos = int(boundary_rows[ww][0])
                bin_size = resolution
                num_key = bp_pos // bin_size
                if m == num_key + 5:
                    current_label = label_T
                    ww += 1
                else:
                    current_label = label_F
            else:
                current_label = label_F

            midx = m
            sub_mat = padded_matrix[midx - win_half : midx + win_half, midx - win_half : midx + win_half]
            sample = np.append(sub_mat.flatten(), current_label)
            all_data.append(sample)

        all_data = np.array(all_data, dtype='float32')
        print(f"chr{chr}, shape of the final sample set: {all_data.shape}")
        np.save(
            f'{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/hic/{cell}_{display_reso}kb_chr{chr}-matrix.npy',
            all_data
        )
        print(f"chr{chr}, saved successfully\n")

